# 03 - Price Sweet-Spot Analysis

**Goal:** Find the optimal price point for each game genre on Steam.

Every indie dev asks the same question: *"How much should I charge for my game?"* Pricing too high scares off buyers, but pricing too low leaves money on the table and can even signal low quality.

In this notebook, we'll dig into the data to find the **"sweet spot"** price for each genre — the price range where games tend to get the most recommendations and estimated revenue.

**What we'll cover:**
1. **Overall price distribution** — what do most games cost?
2. **Price vs. success metrics** — does price correlate with reviews or Metacritic scores?
3. **Genre-level sweet spots** — the best price bracket for each genre
4. **Revenue estimation** — which price range maximizes revenue?
5. **Genre-specific price curves** — visual breakdown for the top genres

Let's find out what the data says.

In [ ]:
# --- Setup: imports and styling ---
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data_loader import load_applications, build_games_with_genres, STEAM_COLORS, CHART_COLORS
from src.utils import setup_matplotlib_style, get_plotly_layout

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

setup_matplotlib_style()
print("All imports loaded successfully!")

In [ ]:
# --- Load data and filter to paid games only ---
# We use build_games_with_genres() so each row is a game-genre pair
games_genres = build_games_with_genres()

# Filter to paid games only (is_free == False AND price > $0)
# Free games would skew our price analysis, so we drop them
paid = games_genres[(games_genres["is_free"] == False) & (games_genres["price_dollars"] > 0)].copy()

print(f"\nTotal game-genre rows: {len(games_genres):,}")
print(f"Paid game-genre rows:  {len(paid):,}")
print(f"Unique paid games:     {paid['appid'].nunique():,}")

# Quick look at price stats
print("\n--- Price Statistics (paid games) ---")
print(f"  Min price:    ${paid['price_dollars'].min():.2f}")
print(f"  Max price:    ${paid['price_dollars'].max():.2f}")
print(f"  Mean price:   ${paid['price_dollars'].mean():.2f}")
print(f"  Median price: ${paid['price_dollars'].median():.2f}")
print(f"  Std dev:      ${paid['price_dollars'].std():.2f}")

## 1. Overall Price Distribution

Before we look at genres, let's understand the big picture — what do most Steam games cost?

In [ ]:
# --- Histogram of game prices (capped at $100 for readability) ---
# Most games are under $100, so we filter to keep the chart clean
price_data = paid[paid["price_dollars"] <= 100].drop_duplicates(subset="appid")
median_price = price_data["price_dollars"].median()

print(f"Showing {len(price_data):,} unique games priced at $100 or less")
print(f"Median price: ${median_price:.2f}")

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=price_data["price_dollars"],
    nbinsx=50,
    marker_color=STEAM_COLORS["light_blue"],
    marker_line=dict(color=STEAM_COLORS["off_white"], width=0.5),
    opacity=0.85,
    name="Games",
))

# Add a vertical line at the median price
fig.add_vline(
    x=median_price,
    line_dash="dash",
    line_color=STEAM_COLORS["accent_orange"],
    line_width=2,
    annotation_text=f"Median: ${median_price:.2f}",
    annotation_position="top right",
    annotation_font=dict(color=STEAM_COLORS["accent_orange"], size=14),
)

layout = get_plotly_layout(
    "Distribution of Steam Game Prices",
    xaxis_title="Price (USD)",
    yaxis_title="Number of Games",
)
fig.update_layout(layout)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# --- Price brackets: group games into meaningful ranges ---
# These brackets are commonly used in the industry
bracket_labels = ["$0-5", "$5-10", "$10-15", "$15-20", "$20-30", "$30-50", "$50-100", "$100+"]
bracket_bins = [0, 5, 10, 15, 20, 30, 50, 100, float("inf")]

# Create the price_bracket column on our paid games dataframe
paid["price_bracket"] = pd.cut(
    paid["price_dollars"],
    bins=bracket_bins,
    labels=bracket_labels,
    right=True,
)

# Count unique games per bracket (drop dupes so multi-genre games count once)
bracket_counts = (
    paid.drop_duplicates(subset="appid")
    .groupby("price_bracket", observed=False)
    .size()
    .reset_index(name="game_count")
)

print("Games per price bracket:")
for _, row in bracket_counts.iterrows():
    print(f"  {row['price_bracket']:>8s}: {row['game_count']:>6,} games")

# Bar chart of game counts per bracket
fig = go.Figure(go.Bar(
    x=bracket_counts["price_bracket"].astype(str),
    y=bracket_counts["game_count"],
    marker_color=CHART_COLORS[:len(bracket_counts)],
    text=bracket_counts["game_count"].apply(lambda x: f"{x:,}"),
    textposition="outside",
    textfont=dict(color=STEAM_COLORS["off_white"]),
))

layout = get_plotly_layout(
    "Number of Games per Price Bracket",
    xaxis_title="Price Bracket",
    yaxis_title="Number of Games",
)
fig.update_layout(layout)
fig.update_layout(showlegend=False)
fig.show()

## 2. Price vs. Success Metrics

Does charging more mean fewer recommendations? Or do expensive games attract more dedicated fanbases? Let's check the relationship between price and two success metrics: **user recommendations** and **Metacritic score**.

In [ ]:
# --- Scatter: Price vs Total Recommendations ---
# Drop duplicates by appid so each game only appears once
scatter_data = paid.drop_duplicates(subset="appid").copy()

# Only include games that have at least 1 recommendation (can't log 0)
scatter_data = scatter_data[scatter_data["recommendations_total"] > 0]
print(f"Plotting {len(scatter_data):,} unique paid games with recommendations")

# Calculate correlation
corr = scatter_data["price_dollars"].corr(scatter_data["recommendations_total"])
print(f"Correlation (price vs recommendations): {corr:.4f}")

# Use log scale for y-axis because recommendations are super skewed
fig = go.Figure(go.Scattergl(
    x=scatter_data["price_dollars"],
    y=scatter_data["recommendations_total"],
    mode="markers",
    marker=dict(
        size=3,
        color=STEAM_COLORS["light_blue"],
        opacity=0.3,
    ),
    name="Games",
))

layout = get_plotly_layout(
    f"Price vs. Recommendations (r = {corr:.3f})",
    xaxis_title="Price (USD)",
    yaxis_title="Total Recommendations (log scale)",
)
fig.update_layout(layout)
fig.update_yaxes(type="log")  # Log scale so we can see the spread
fig.update_xaxes(range=[0, 100])  # Focus on $0-100 range
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# --- Scatter: Price vs Metacritic Score ---
# Only games that actually have a Metacritic score
meta_data = paid.drop_duplicates(subset="appid").copy()
meta_data = meta_data[meta_data["metacritic_score"].notna() & (meta_data["metacritic_score"] > 0)]
print(f"Games with Metacritic scores: {len(meta_data):,}")

# Calculate correlation
corr_meta = meta_data["price_dollars"].corr(meta_data["metacritic_score"])
print(f"Correlation (price vs metacritic): {corr_meta:.4f}")

fig = go.Figure(go.Scattergl(
    x=meta_data["price_dollars"],
    y=meta_data["metacritic_score"],
    mode="markers",
    marker=dict(
        size=5,
        color=STEAM_COLORS["accent_green"],
        opacity=0.4,
    ),
    name="Games",
))

layout = get_plotly_layout(
    f"Price vs. Metacritic Score (r = {corr_meta:.3f})",
    xaxis_title="Price (USD)",
    yaxis_title="Metacritic Score",
)
fig.update_layout(layout)
fig.update_xaxes(range=[0, 80])  # Focus on the main price range
fig.update_layout(showlegend=False)
fig.show()

## 3. Sweet-Spot Analysis Per Genre

Now the fun part — let's find the **best price bracket for each genre**. We'll look at which price range gets the most recommendations on average, since that's our best proxy for player engagement.

In [ ]:
# --- Find top 10 genres by game count ---
top_genres = (
    paid.groupby("genre_name")["appid"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .index
    .tolist()
)

print("Top 10 genres by game count:")
for i, genre in enumerate(top_genres, 1):
    count = paid[paid["genre_name"] == genre]["appid"].nunique()
    print(f"  {i:>2}. {genre:<25s} ({count:,} games)")

# Filter to just these top genres
top_paid = paid[paid["genre_name"].isin(top_genres)].copy()

# Group by genre and price bracket, calculate avg recommendations and game count
genre_bracket = (
    top_paid
    .groupby(["genre_name", "price_bracket"], observed=False)
    .agg(
        avg_recommendations=("recommendations_total", "mean"),
        game_count=("appid", "nunique"),
    )
    .reset_index()
)

# Filter out brackets with too few games (less than 10) — not statistically meaningful
genre_bracket_filtered = genre_bracket[genre_bracket["game_count"] >= 10].copy()

print(f"\nTotal genre-bracket combos: {len(genre_bracket)}")
print(f"After filtering (>= 10 games): {len(genre_bracket_filtered)}")

In [ ]:
# --- Heatmap: Avg Recommendations by Genre x Price Bracket ---
# Pivot so genres are rows and price brackets are columns
heatmap_data = genre_bracket_filtered.pivot_table(
    index="genre_name",
    columns="price_bracket",
    values="avg_recommendations",
    observed=False,
)

# Reorder genres by total avg recommendations (highest at top)
genre_order = heatmap_data.mean(axis=1).sort_values(ascending=True).index.tolist()
heatmap_data = heatmap_data.loc[genre_order]

# Build the text annotations (rounded values for readability)
text_vals = heatmap_data.round(0).fillna("").astype(str)
text_vals = text_vals.replace("nan", "")

fig = go.Figure(go.Heatmap(
    z=heatmap_data.values,
    x=[str(c) for c in heatmap_data.columns],
    y=heatmap_data.index.tolist(),
    text=text_vals.values,
    texttemplate="%{text}",
    textfont=dict(size=10, color="white"),
    colorscale=[
        [0, STEAM_COLORS["dark_blue"]],
        [0.5, STEAM_COLORS["medium_blue"]],
        [1, STEAM_COLORS["light_blue"]],
    ],
    colorbar=dict(
        title=dict(text="Avg Recs", font=dict(color=STEAM_COLORS["off_white"])),
        tickfont=dict(color=STEAM_COLORS["off_white"]),
    ),
    hoverongaps=False,
))

layout = get_plotly_layout(
    "Average Recommendations by Genre & Price Bracket",
    xaxis_title="Price Bracket",
    yaxis_title="Genre",
)
fig.update_layout(layout)
fig.update_layout(height=500)
fig.show()

In [ ]:
# --- Find the BEST price bracket per genre ---
# "Best" = highest average recommendations (our proxy for success)
best_brackets = (
    genre_bracket_filtered
    .loc[genre_bracket_filtered.groupby("genre_name")["avg_recommendations"].idxmax()]
    .sort_values("avg_recommendations", ascending=False)
    .reset_index(drop=True)
)

print("=" * 65)
print("  SWEET-SPOT PRICE BRACKET PER GENRE (by avg recommendations)")
print("=" * 65)
print(f"  {'Genre':<22s} {'Best Bracket':>13s} {'Avg Recs':>10s} {'# Games':>8s}")
print("-" * 65)

for _, row in best_brackets.iterrows():
    print(
        f"  {row['genre_name']:<22s}"
        f" {str(row['price_bracket']):>13s}"
        f" {row['avg_recommendations']:>10,.0f}"
        f" {row['game_count']:>8,}"
    )

print("=" * 65)
print("\nNote: Only includes brackets with 10+ games for reliability.")

## 4. Revenue Estimation

Recommendations are great, but developers also care about **money**. Let's estimate revenue using a simple formula:

> `estimated_revenue = price_dollars * recommendations_total`

This is a rough proxy — not every recommendation means a sale, but it gives us a sense of which price brackets generate the most value.

In [ ]:
# --- Revenue estimation per price bracket ---
# Calculate estimated revenue for each game
paid["estimated_revenue"] = paid["price_dollars"] * paid["recommendations_total"]

# Group by price bracket (unique games only)
revenue_by_bracket = (
    paid.drop_duplicates(subset="appid")
    .groupby("price_bracket", observed=False)
    .agg(
        avg_revenue=("estimated_revenue", "mean"),
        median_revenue=("estimated_revenue", "median"),
        game_count=("appid", "nunique"),
    )
    .reset_index()
)

print("Estimated Revenue by Price Bracket:")
print(f"  {'Bracket':>10s} {'Avg Revenue':>14s} {'Median Revenue':>16s} {'# Games':>8s}")
print("-" * 55)
for _, row in revenue_by_bracket.iterrows():
    print(
        f"  {str(row['price_bracket']):>10s}"
        f"  ${row['avg_revenue']:>12,.0f}"
        f"  ${row['median_revenue']:>14,.0f}"
        f"  {row['game_count']:>7,}"
    )

# Grouped bar chart: avg vs median revenue
fig = go.Figure()

fig.add_trace(go.Bar(
    x=revenue_by_bracket["price_bracket"].astype(str),
    y=revenue_by_bracket["avg_revenue"],
    name="Average Revenue",
    marker_color=STEAM_COLORS["light_blue"],
))

fig.add_trace(go.Bar(
    x=revenue_by_bracket["price_bracket"].astype(str),
    y=revenue_by_bracket["median_revenue"],
    name="Median Revenue",
    marker_color=STEAM_COLORS["accent_orange"],
))

layout = get_plotly_layout(
    "Estimated Revenue by Price Bracket (Avg vs Median)",
    xaxis_title="Price Bracket",
    yaxis_title="Estimated Revenue (USD)",
)
fig.update_layout(layout)
fig.update_layout(barmode="group")
fig.show()

In [ ]:
# --- Box plot of estimated revenue per bracket ---
# Filter to < $10M for readability (outliers would crush the chart)
revenue_box_data = (
    paid.drop_duplicates(subset="appid")
    .copy()
)
revenue_box_data = revenue_box_data[revenue_box_data["estimated_revenue"] < 10_000_000]

print(f"Showing {len(revenue_box_data):,} games with estimated revenue < $10M")

fig = go.Figure()

# Add a box for each price bracket
for i, bracket in enumerate(bracket_labels):
    bracket_data = revenue_box_data[revenue_box_data["price_bracket"] == bracket]
    if len(bracket_data) > 0:
        fig.add_trace(go.Box(
            y=bracket_data["estimated_revenue"],
            name=bracket,
            marker_color=CHART_COLORS[i % len(CHART_COLORS)],
            boxmean=True,  # Show mean as a dashed line
        ))

layout = get_plotly_layout(
    "Revenue Distribution by Price Bracket (< $10M)",
    xaxis_title="Price Bracket",
    yaxis_title="Estimated Revenue (USD)",
)
fig.update_layout(layout)
fig.update_layout(showlegend=False)
fig.show()

## 5. Genre-Specific Price Curves

Different genres have different audiences — a strategy game fan might pay more than a casual puzzle player. Let's look at the price-vs-recommendations curve for each of the top 5 genres individually.

In [ ]:
# --- Subplots: price curves for top 5 genres ---
top_5_genres = top_genres[:5]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=top_5_genres,
    horizontal_spacing=0.1,
    vertical_spacing=0.15,
)

for idx, genre in enumerate(top_5_genres):
    row = idx // 3 + 1
    col = idx % 3 + 1

    # Get data for this genre
    genre_data = genre_bracket_filtered[genre_bracket_filtered["genre_name"] == genre].copy()

    # Sort by price bracket order
    genre_data["price_bracket"] = pd.Categorical(
        genre_data["price_bracket"], categories=bracket_labels, ordered=True
    )
    genre_data = genre_data.sort_values("price_bracket")

    fig.add_trace(
        go.Bar(
            x=genre_data["price_bracket"].astype(str),
            y=genre_data["avg_recommendations"],
            marker_color=CHART_COLORS[idx % len(CHART_COLORS)],
            name=genre,
            showlegend=False,
        ),
        row=row,
        col=col,
    )

# Apply Steam dark theme to the whole figure
fig.update_layout(
    title=dict(
        text="Average Recommendations per Price Bracket — Top 5 Genres",
        font=dict(size=18, color=STEAM_COLORS["off_white"]),
        x=0.5,
    ),
    paper_bgcolor=STEAM_COLORS["dark_blue"],
    plot_bgcolor=STEAM_COLORS["medium_blue"],
    font=dict(color=STEAM_COLORS["off_white"], size=10),
    height=550,
    showlegend=False,
)

# Style all axes
fig.update_xaxes(
    gridcolor="rgba(198, 213, 224, 0.15)",
    tickangle=45,
    tickfont=dict(size=9),
)
fig.update_yaxes(
    gridcolor="rgba(198, 213, 224, 0.15)",
    title_text="Avg Recs",
    title_font=dict(size=10),
)

# Update subplot title colors
for annotation in fig["layout"]["annotations"]:
    annotation["font"] = dict(size=13, color=STEAM_COLORS["off_white"])

fig.show()

print("\nEach subplot shows how average recommendations change across price brackets.")
print("Look for the tallest bar in each genre — that's the sweet spot!")

## Summary & Key Findings

### What the data tells us:

1. **Most Steam games are cheap.** The median price clusters in the low-teens range, with a huge number of games priced under $10. The market is crowded at the bottom.

2. **Price alone doesn't determine success.** The correlation between price and recommendations is weak — a great cheap game can outperform an expensive one, and vice versa.

3. **The sweet spot varies by genre.** Different genres have different optimal price points. Some genres (like RPGs and Strategy) tend to perform better at higher price brackets, while others (like Casual and Indie) do well at lower prices.

4. **Median revenue tells a different story than average.** A few mega-hits in each bracket pull the average way up, but the median shows what a "typical" game earns. The gap between average and median revenue is huge — most games don't come close to the average.

### Pricing recommendations for indie devs:

- **Don't race to the bottom.** Pricing at $1-2 rarely leads to more recommendations or revenue. Players often associate very low prices with low quality.
- **Match your genre's expectations.** Check the sweet-spot table above for your genre and price accordingly.
- **Consider the $10-20 range.** For most genres, this range strikes a good balance between accessibility and perceived value.
- **Focus on quality over pricing tricks.** The weak price-to-success correlation means that making a great game matters more than nailing the perfect price.

---
*Next up: Notebook 04 will explore time-based trends — how has the Steam market changed over the years?*